> Check the signals in some clusters

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import pandas as pd

import jax.random as jr

import plotly
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go

import trapallada.analysis

from IPython.display import display, Markdown

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

The base directory

In [ ]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

# Input Files

Widget to choose a *sub*directory with results

In [ ]:
search_dir = base_dir / 'results' / 'normalization' / 'normalized_rate'
assert search_dir.exists()
combo = trapallada.analysis.make_combo_widget(search_dir)
display(combo)

Particular *sub*directory with the trained model. It should be a `str` so that parameters can be injected with papermill.

In [ ]:
model_dir = search_dir / combo.value
# ===============
# model_subdir = 'results/normalization/normalized_rate/fft_vanilla_x_sd_00025'
# model_subdir = 'results/normalization/normalized_rate/vanilla_x_sd_015_z_40'
# ===============
# model_subdir = 'results/normalization/clean_0.9/vanilla'
# model_subdir = 'results/normalization/clean_0.9/vanilla_x_sd_015'
# ===============
# model_subdir = 'results/normalization/clipped/vanilla_x_sd_015'
# model_subdir = 'results/normalization/clipped/fft_vanilla_x_sd_00025'
# model_subdir = 'results/normalization/count_rate_5/vanilla_x_sd_015'
# ===============
model_subdir = "results/masked/min_count_rate_002/vanilla_x_sd_015/"

Just in case

In [ ]:
model_dir = base_dir / model_subdir
display(model_dir)
assert model_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(model_dir, analysis_also=True)

In [ ]:
analysis_df.head()

For the sake of convenience

In [ ]:
# pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

The clustering (either *in projected space* or *in z*) to be analyzed

In [ ]:
available_clusterings = ['cluster in z', 'cluster in projected space']

Seemingly, *parquet* format dismisses `Categorical` data type

In [ ]:
available_clusterings = ['cluster in z', 'cluster in projected space']
analysis_df[available_clusterings].dtypes

It is manually fixed

In [ ]:
for c in available_clusterings:
    analysis_df[c] = analysis_df[c].astype('category')

The number of objects for every *type*

In [ ]:
analysis_df['main_type'].value_counts()

In [ ]:
on_plot_hover_columns = ['index', 'source', 'main_type'] + available_clusterings
on_plot_hover_columns

# Training

Curves for the training/validation
- loss, 
- log-likelihood, and
- KL divergence

In [ ]:
trapallada.analysis.metric_chart(results_df, 'loss') | \
    trapallada.analysis.metric_chart(results_df, 'loglikelihood') | \
        trapallada.analysis.metric_chart(results_df, 'kl')

# Latent space

Mean and standard deviation for every variable in the latent space

In [ ]:
latents_df = analysis_df.filter(like='z_').agg(['mean', 'std']).T
# latents_df.head()
px.scatter(latents_df, x=latents_df.index, y='mean', error_y='std')

# Clusters

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=tuple(available_clusterings))

for i_c, c in enumerate(available_clusterings):
    n_samples_per_cluster = analysis_df[c].value_counts()
    n_clusters = len(n_samples_per_cluster)
    fig.add_trace(go.Bar(x=n_samples_per_cluster.index, y=n_samples_per_cluster.values, name=c),row=1, col=i_c+1)
    print(f'{n_clusters} {c}')

fig.update_layout(title_text="Clusters", showlegend=False)
fig.show()


The directory with the full signals is inferred

In [ ]:
full_signals_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent / 'full_signals'

The windows in the projected space, colored according to its
- cluster in the latent space (`z`)

In [ ]:
trapallada.analysis.TSNE(analysis_df, tsne_columns, full_signals_dir, on_plot_hover_columns).make_fig(color_column='cluster in z')

(some signals)

In [ ]:
# trapallada.analysis.plot_samples_from_cluster(analysis_df, full_signals_dir, cluster_column='cluster in z')


- cluster in the projected space

In [ ]:
trapallada.analysis.TSNE(analysis_df, tsne_columns, full_signals_dir, on_plot_hover_columns).make_fig(color_column='cluster in projected space')

(some signals)

In [ ]:
# trapallada.analysis.plot_samples_from_cluster(analysis_df, full_signals_dir, cluster_column='cluster in projected space')

# Signals by `main_type`

The (selected beforehand) *signals of interest*

In [ ]:
signals_of_interest_df = analysis_df[
    analysis_df['signal'].isin(
        map(lambda x: x[0], trapallada.analysis.get_signals_of_interest())
    )
]
signals_of_interest_df.head()

Empty `main_type`s are set to `UNKNOWN`

In [ ]:
analysis_df['main_type'].fillna('UNKNOWN', inplace=True)

In order to have fixed axes

In [ ]:
xrange, yrange = list(zip(analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()))
xrange, yrange

One can **double-click** a `main_type` in the *Legend* to only show the corresponding curves

In [ ]:
analysis_df['main_type'].unique().shape

In [ ]:
analysis_df['signal']

All the windows are plotted, and some of them, considered *of interest*, are highlighted.

In [ ]:
tsne_plot = trapallada.analysis.TSNE(analysis_df, tsne_columns, full_signals_dir, on_plot_hover_columns)
fig = tsne_plot.make_fig()
tsne_plot.highlight_windows_of_interest()
fig.show()

# Zooming in

A square centered at the origin.

In [ ]:
regions = [
    {
        'x_range': [-10, 10],
        'y_range': [-10, 10],
    },
]


In [ ]:
regions += [
    # {
    #     'x_range': [55, 75],
    #     'y_range': [-10, 15],
    # },
]

regions_file = model_dir / 'regions.yaml'

if regions_file.exists():
    regions = trapallada.analysis.load_regions(regions_file)
    print(f'Loaded {len(regions)} regions')


labels = [f'Region {i}' for i in range(len(regions))]

trapallada.analysis.make_save_button_widget(regions, regions_file)

In [ ]:
tsne_plot.inspect_regions(regions, labels, n_samples=5)
# tsne_plot.inspect_regions(regions, labels, dont_plot_signals=True)

# Some selected signals

In [ ]:
for i in [25160]:
    display(trapallada.analysis.check_window(i, model, ds, analysis_df, full_signals_dir))